# Generate Gmail token.json for the Gmail Cleanup AI Agent

Use this notebook to generate a `token.json` file for your own Gmail account.

## What you need
- A Google Cloud project with Gmail API enabled
- OAuth Desktop App credentials downloaded as `credentials.json`

## What this notebook does
1. Upload `credentials.json`
2. Open a Google consent link
3. Paste back the authorization code
4. Generate `token.json`
5. Download `token.json`

Then upload `token.json` into the root of the GitHub Codespaces project.


In [ ]:
!pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

In [ ]:
from google.colab import files

uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

In [ ]:
import os

uploaded_files = os.listdir()

if 'credentials.json' not in uploaded_files:
    for f in uploaded_files:
        if f.endswith('.json'):
            os.rename(f, 'credentials.json')
            print(f'Renamed {f} -> credentials.json')
            break

print(os.listdir())

In [ ]:
from google_auth_oauthlib.flow import Flow

SCOPES = ['https://www.googleapis.com/auth/gmail.modify']

flow = Flow.from_client_secrets_file(
    'credentials.json',
    scopes=SCOPES,
    redirect_uri='urn:ietf:wg:oauth:2.0:oob'
)

auth_url, state = flow.authorization_url(
    access_type='offline',
    prompt='consent'
)

print('Open this URL in your browser and complete login:\n')
print(auth_url)
print('\nState:\n', state)

In [ ]:
code = input('Paste the authorization code here: ').strip()

flow.fetch_token(code=code)
creds = flow.credentials

with open('token.json', 'w') as f:
    f.write(creds.to_json())

print('token.json created successfully.')

In [ ]:
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials

creds = Credentials.from_authorized_user_file('token.json', SCOPES)
service = build('gmail', 'v1', credentials=creds)

profile = service.users().getProfile(userId='me').execute()
print('Authenticated Gmail address:', profile['emailAddress'])
print('Total messages:', profile['messagesTotal'])

In [ ]:
from google.colab import files
files.download('token.json')